In [1]:
# =============================================================================
# ETA Prediction — Step 8: Hyperparameter Tuning (HalvingRandomSearchCV)
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.experimental import enable_halving_search_cv  # noqa: bắt buộc để bật HalvingRandomSearchCV
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import randint, uniform

import lightgbm as lgb
import xgboost as xgb

print("Step 8 Started - Hyperparameter Tuning")

Mounted at /content/drive
Step 8 Started - Hyperparameter Tuning


In [2]:
# ====================== 8.1 LOAD DATA (Checkpoints_v3) ======================
CKPT_DIR = "/content/drive/MyDrive/LaDe/Checkpoints_v3"

X_train = joblib.load(f"{CKPT_DIR}/X_train.pkl")
X_val   = joblib.load(f"{CKPT_DIR}/X_val.pkl")
X_test  = joblib.load(f"{CKPT_DIR}/X_test.pkl")

y_train = joblib.load(f"{CKPT_DIR}/y_train.pkl")
y_val   = joblib.load(f"{CKPT_DIR}/y_val.pkl")
y_test  = joblib.load(f"{CKPT_DIR}/y_test.pkl")

y_train_min = joblib.load(f"{CKPT_DIR}/y_train_min.pkl")
y_val_min   = joblib.load(f"{CKPT_DIR}/y_val_min.pkl")
y_test_min  = joblib.load(f"{CKPT_DIR}/y_test_min.pkl")

final_features = joblib.load(f"{CKPT_DIR}/final_features.pkl")

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Features: {len(final_features)}")

Train: 672,690 | Val: 224,230 | Test: 224,231
Features: 24


In [3]:
# ====================== 8.2 BASELINE (từ Step 6, để so sánh) ======================
# Số liệu baseline đã train ở Step 6 (full param mặc định + early stopping),
# chép thủ công vào đây để in bảng so sánh cuối cùng — không train lại.
baseline_scores = {
    "LightGBM": {"mae": 47.85, "rmse": 87.18},
    "XGBoost":  {"mae": 43.98, "rmse": 84.22},
}
print("Baseline (Step 6):", baseline_scores)

Baseline (Step 6): {'LightGBM': {'mae': 47.85, 'rmse': 87.18}, 'XGBoost': {'mae': 43.98, 'rmse': 84.22}}


In [4]:
# ====================== 8.3 EVALUATION HELPER ======================
def evaluate(y_true_min, y_pred_log, name="Model"):
    y_pred_min = np.expm1(y_pred_log)
    mae = mean_absolute_error(y_true_min, y_pred_min)
    rmse = np.sqrt(mean_squared_error(y_true_min, y_pred_min))
    print(f"\n {name:25s} | MAE: {mae:6.2f} mins | RMSE: {rmse:6.2f}")
    return mae, rmse

In [5]:
# ====================== 8.4 LIGHTGBM — HALVING RANDOM SEARCH ======================
# Lưu ý từ lịch sử debug: KHÔNG đặt n_jobs=-1 đồng thời ở cả search lẫn estimator
# (gây PicklingError trong Colab multiprocessing). Ở đây: estimator n_jobs=-1,
# search n_jobs=1 (sequential candidates, mỗi candidate tự song song hoá nội bộ).

lgb_param_dist = {
    'num_leaves':        randint(31, 256),
    'max_depth':         randint(4, 16),
    'learning_rate':     uniform(0.01, 0.19),       # 0.01 - 0.20
    'feature_fraction':  uniform(0.6, 0.4),         # 0.6 - 1.0
    'bagging_fraction':  uniform(0.6, 0.4),
    'bagging_freq':      randint(1, 10),
    'min_child_samples': randint(5, 100),
    'reg_alpha':         uniform(0.0, 5.0),
    'reg_lambda':        uniform(0.0, 5.0),
}

lgb_estimator = lgb.LGBMRegressor(
    objective='regression',
    metric='mae',
    n_estimators=600,          # cap thấp khi search (search chỉ để tìm vùng tham số tốt)
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# min_resources = 30% train set (theo kinh nghiệm XGBoost stalling ở Step 6),
# factor=4 để halving co nhanh, tránh tốn thời gian ở vòng đầu.
lgb_search = HalvingRandomSearchCV(
    estimator=lgb_estimator,
    param_distributions=lgb_param_dist,
    resource='n_samples',
    min_resources=int(0.3 * len(X_train)),
    factor=4,
    n_candidates='exhaust',
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=1,
    verbose=1,
)

print("Đang search LightGBM (có thể mất nhiều phút)...")
lgb_search.fit(X_train, y_train)

print("\nBest params (LightGBM):")
print(lgb_search.best_params_)
print(f"Best CV score (neg MAE, log-scale): {lgb_search.best_score_:.4f}")

Đang search LightGBM (có thể mất nhiều phút)...
n_iterations: 1
n_required_iterations: 1
n_possible_iterations: 1
min_resources_: 201807
max_resources_: 672690
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 3
n_resources: 201807
Fitting 3 folds for each of 3 candidates, totalling 9 fits

Best params (LightGBM):
{'bagging_fraction': np.float64(0.8469926038510867), 'bagging_freq': 6, 'feature_fraction': np.float64(0.6028265220878869), 'learning_rate': np.float64(0.014381860757868993), 'max_depth': 14, 'min_child_samples': 63, 'num_leaves': 200, 'reg_alpha': np.float64(0.23332831606807714), 'reg_lambda': np.float64(4.8687775942072955)}
Best CV score (neg MAE, log-scale): -0.4641


In [6]:
# ====================== 8.5 XGBOOST — HALVING RANDOM SEARCH ======================
xgb_param_dist = {
    'max_depth':         randint(4, 14),
    'learning_rate':     uniform(0.01, 0.19),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
    'min_child_weight':  randint(1, 20),
    'reg_alpha':         uniform(0.0, 5.0),
    'reg_lambda':        uniform(0.0, 5.0),
    'gamma':             uniform(0.0, 5.0),
}

xgb_estimator = xgb.XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=600,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',   # nhanh hơn cho data lớn, tránh lặp lại sự cố "XGBoost stalling"
)

xgb_search = HalvingRandomSearchCV(
    estimator=xgb_estimator,
    param_distributions=xgb_param_dist,
    resource='n_samples',
    min_resources=int(0.3 * len(X_train)),
    factor=4,
    n_candidates='exhaust',
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=1,
    verbose=1,
)

print("Đang search XGBoost (có thể mất nhiều phút)...")
xgb_search.fit(X_train, y_train)

print("\nBest params (XGBoost):")
print(xgb_search.best_params_)
print(f"Best CV score (neg MAE, log-scale): {xgb_search.best_score_:.4f}")

Đang search XGBoost (có thể mất nhiều phút)...
n_iterations: 1
n_required_iterations: 1
n_possible_iterations: 1
min_resources_: 201807
max_resources_: 672690
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 3
n_resources: 201807
Fitting 3 folds for each of 3 candidates, totalling 9 fits

Best params (XGBoost):
{'colsample_bytree': np.float64(0.996884623716487), 'gamma': np.float64(3.0874075481385828), 'learning_rate': np.float64(0.12621410049277337), 'max_depth': 12, 'min_child_weight': 17, 'reg_alpha': np.float64(2.6238733012919457), 'reg_lambda': np.float64(1.9993048585762774), 'subsample': np.float64(0.6186662652854461)}
Best CV score (neg MAE, log-scale): -0.4446


In [8]:
# ====================== 8.6 RETRAIN FINAL MODELS VỚI BEST PARAMS + EARLY STOPPING ======================
# Search ở trên chỉ dùng n_estimators=600 cố định để search nhanh.
# Bây giờ train lại với early stopping thật (giống Step 6) để lấy số vòng tối ưu.

print("Retrain LightGBM với best params...")
best_lgb_params = dict(lgb_search.best_params_)
best_lgb_params.update({
    'objective': 'regression',
    'metric': 'mae',
    'random_state': 42,
    'verbose': -1,
})

train_set = lgb.Dataset(X_train, label=y_train)
val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

lgb_model_tuned = lgb.train(
    best_lgb_params, train_set,
    num_boost_round=5000,
    valid_sets=[val_set],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)
print(f"LightGBM (tuned) stopped at iteration: {lgb_model_tuned.best_iteration}")

pred_lgb_tuned = lgb_model_tuned.predict(X_test, num_iteration=lgb_model_tuned.best_iteration)
mae_lgb_tuned, rmse_lgb_tuned = evaluate(y_test_min, pred_lgb_tuned, "LightGBM (tuned)")

Retrain LightGBM với best params...
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 0.502419
[400]	valid_0's l1: 0.470101
[600]	valid_0's l1: 0.461721
[800]	valid_0's l1: 0.457919
[1000]	valid_0's l1: 0.456059
[1200]	valid_0's l1: 0.454486
[1400]	valid_0's l1: 0.453382
[1600]	valid_0's l1: 0.452488
[1800]	valid_0's l1: 0.451598
[2000]	valid_0's l1: 0.450836
[2200]	valid_0's l1: 0.450083
[2400]	valid_0's l1: 0.449424
[2600]	valid_0's l1: 0.44888
[2800]	valid_0's l1: 0.448386
[3000]	valid_0's l1: 0.447911
[3200]	valid_0's l1: 0.447499
[3400]	valid_0's l1: 0.447096
[3600]	valid_0's l1: 0.44677
[3800]	valid_0's l1: 0.446474
[4000]	valid_0's l1: 0.446228
[4200]	valid_0's l1: 0.445947
[4400]	valid_0's l1: 0.445754
[4600]	valid_0's l1: 0.445484
[4800]	valid_0's l1: 0.445267
[5000]	valid_0's l1: 0.445095
Did not meet early stopping. Best iteration is:
[4999]	valid_0's l1: 0.445095
LightGBM (tuned) stopped at iteration: 4999

 LightGBM (tuned)          | MAE:  

In [9]:
print("Retrain XGBoost với best params...")
best_xgb_params = dict(xgb_search.best_params_)
best_xgb_params.update({
    'objective': 'reg:absoluteerror',
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist',
})

xgb_model_tuned = xgb.XGBRegressor(
    n_estimators=5000,
    early_stopping_rounds=100,
    **best_xgb_params
)
xgb_model_tuned.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"XGBoost (tuned) stopped at iteration: {xgb_model_tuned.best_iteration}")

pred_xgb_tuned = xgb_model_tuned.predict(X_test, iteration_range=(0, xgb_model_tuned.best_iteration + 1))
mae_xgb_tuned, rmse_xgb_tuned = evaluate(y_test_min, pred_xgb_tuned, "XGBoost (tuned)")

Retrain XGBoost với best params...
XGBoost (tuned) stopped at iteration: 782

 XGBoost (tuned)           | MAE:  43.81 mins | RMSE:  84.05


In [10]:
# ====================== 8.7 SO SÁNH BASELINE VS TUNED ======================
comparison = pd.DataFrame([
    {"model": "LightGBM", "version": "baseline", "mae": baseline_scores["LightGBM"]["mae"], "rmse": baseline_scores["LightGBM"]["rmse"]},
    {"model": "LightGBM", "version": "tuned",    "mae": mae_lgb_tuned, "rmse": rmse_lgb_tuned},
    {"model": "XGBoost",  "version": "baseline", "mae": baseline_scores["XGBoost"]["mae"],  "rmse": baseline_scores["XGBoost"]["rmse"]},
    {"model": "XGBoost",  "version": "tuned",    "mae": mae_xgb_tuned, "rmse": rmse_xgb_tuned},
])
comparison["mae_improvement_%"] = comparison.groupby("model")["mae"].transform(
    lambda s: (s.iloc[0] - s) / s.iloc[0] * 100
)
print(comparison.round(2))

# Chọn model tốt nhất theo test MAE để dùng cho Step 9
best_model_name = "LightGBM" if mae_lgb_tuned < mae_xgb_tuned else "XGBoost"
print(f"\n>>> Model được chọn cho Step 9: {best_model_name}")

      model   version    mae   rmse  mae_improvement_%
0  LightGBM  baseline  47.85  87.18               0.00
1  LightGBM     tuned  48.38  87.20              -1.11
2   XGBoost  baseline  43.98  84.22               0.00
3   XGBoost     tuned  43.81  84.05               0.38

>>> Model được chọn cho Step 9: XGBoost


In [11]:
# ====================== 8.8 SAVE TUNED MODELS (Checkpoints_v4) ======================
CKPT_DIR_V4 = "/content/drive/MyDrive/LaDe/Checkpoints_v4"
os.makedirs(CKPT_DIR_V4, exist_ok=True)

joblib.dump(lgb_model_tuned, f"{CKPT_DIR_V4}/lgb_model_tuned.pkl")
joblib.dump(xgb_model_tuned, f"{CKPT_DIR_V4}/xgb_model_tuned.pkl")

joblib.dump(best_lgb_params, f"{CKPT_DIR_V4}/best_lgb_params.pkl")
joblib.dump(best_xgb_params, f"{CKPT_DIR_V4}/best_xgb_params.pkl")

joblib.dump(best_model_name, f"{CKPT_DIR_V4}/best_model_name.pkl")
comparison.to_csv(f"{CKPT_DIR_V4}/tuning_comparison.csv", index=False)

print(f"Đã lưu model + params vào {CKPT_DIR_V4}")
print("LƯU Ý: Checkpoints_v4 chỉ chứa model/params mới. Data (X_*, y_*), final_features,")
print("te_maps, courier_stats vẫn nằm ở Checkpoints_v3 — Step 9 sẽ load từ cả hai nơi.")

Đã lưu model + params vào /content/drive/MyDrive/LaDe/Checkpoints_v4
LƯU Ý: Checkpoints_v4 chỉ chứa model/params mới. Data (X_*, y_*), final_features,
te_maps, courier_stats vẫn nằm ở Checkpoints_v3 — Step 9 sẽ load từ cả hai nơi.
